# 駅のまわりを広げると、似た街は変わる？
500・600・700・800mは駅から東西南北への半幅です。一辺は1.0〜1.6km。最適な商圏を決めず、分析条件への感度を調べます。

[固定仕様](../docs/aoi-sensitivity-design.md) / [方法](../docs/aoi-sensitivity.md) / [検証](../docs/aoi-sensitivity-validation.md)

In [ ]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks': os.chdir('..')
from geoai_open_lab.aoi_sensitivity import run
from geoai_open_lab.aoi_sensitivity_figures import figures
from IPython.display import display, SVG
import pandas as pd

## 1. 同じ入力・同じ候補集合で比較
保存済み入力だけで実行します。500mの特徴量・旧順位・クラスタを照合し、主比較は800mで重なる候補を全条件で除外します。Aは8特徴、Bは固定961カテゴリTF-IDF、CはSVD64＋L2。各範囲で学習し直すため、Cの軸を直接比較しません。

In [ ]:
summary, manifest, out = run()
paths = figures()
display(summary.round(3))

## 2. 吉祥寺に似ている駅
ランキングの距離をA/B/C間で直接比較しません。共通数は駅グループIDで数えます。

In [ ]:
rankings = pd.read_csv(out/'rankings.csv')
display(rankings[(rankings['rank'] <= 10) & (rankings.half_width_m == 800)].pivot(index='rank', columns='mode', values='station_name'))
display(SVG(filename=str(out/'target_rankings.svg')))

## 3. 全駅の分布と3群分類
平均だけでなく分布を見ます。ARIは分類の一致度で、正解率ではありません。群番号は最大一致で対応させます。

In [ ]:
display(SVG(filename=str(out/'common_distribution.svg')))
display(SVG(filename=str(out/'cluster_changes.svg')))

## 4. 限界
AOI拡張と同時にIDF・SVD・Aの標準化を再学習しています。Aの250m格子は位置を固定し、端の部分区画は実面積を使います。小さな部分区画の最大密度への影響も含む感度です。近接駅は同じPOIを共有し、独立した街ではありません。増えたカテゴリは順位変化の原因と断定しません。

正解となる街の範囲・類似ラベルはなく、範囲を広げるほど精度が上がるとは言えません。円・徒歩圏との比較は未実施です。